<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/3_2_Cross%E2%80%91Validation%2C_Bias%E2%80%91Variance_Tradeoff%2C_and_Hyperparameter_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part III. Evaluation Toolkit and Model Selection. 9. Cross‑Validation,8Bias‑Variance Tradeoff, and Hyperparameter Selection

### 8.1. Bias‑Variance Tradeoff: теоретический фундамент

В главе 7 мы научились измерять качество моделей — от accuracy и \(F_1\) до AUC и MSE. Но почему одна модель работает лучше другой на новых данных? Почему добавление признаков иногда улучшает, а иногда ухудшает предсказания? Ответ кроется в фундаментальном разложении ошибки на смещение, дисперсию и неустранимый шум. Понимание этого компромисса — ключ к осознанному выбору сложности модели и стратегии валидации.

---

#### 1. Постановка задачи и ожидаемая ошибка

Рассмотрим задачу регрессии. Предположим, что истинная зависимость между признаками \(\mathbf{x} \in \mathbb{R}^d\) и откликом \(y \in \mathbb{R}\) имеет вид

\[
y = f(\mathbf{x}) + \varepsilon,
\]

где \(f(\mathbf{x})\) — детерминированная функция, а \(\varepsilon\) — случайный шум, обладающий свойствами

\[
\mathbb{E}[\varepsilon] = 0, \quad \mathrm{Var}(\varepsilon) = \sigma^2,
\]

и \(\varepsilon\) не зависит от \(\mathbf{x}\). Дисперсия \(\sigma^2\) отражает принципиально неустранимую стохастическую составляющую (ошибки измерения, влияние ненаблюдаемых факторов).

Модель обучается на конкретной конечной выборке \(\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n\) и даёт предсказательную функцию \(\hat{f}(\mathbf{x}; \mathcal{D})\). Для фиксированной новой точки \(\mathbf{x}_0\) качество модели естественно оценивать **ожидаемой квадратичной ошибкой предсказания**, усреднённой как по случайности обучающей выборки, так и по шуму:

\[
\text{Err}(\mathbf{x}_0) = \mathbb{E}_{\mathcal{D},\, \varepsilon}\Bigl[ \bigl( y_0 - \hat{f}(\mathbf{x}_0; \mathcal{D}) \bigr)^2 \Bigr],
\]

где \(y_0 = f(\mathbf{x}_0) + \varepsilon_0\), а \(\varepsilon_0\) — новая реализация шума, независимая от \(\mathcal{D}\). Именно эта величина определяет, насколько хорошо модель будет работать в среднем, если бы мы многократно собирали новые обучающие выборки.

---

#### 2. Разложение ожидаемой квадратичной ошибки

Проведём вывод разложения \(\text{Err}(\mathbf{x}_0)\) на три интерпретируемые составляющие. Для краткости будем писать \(\hat{f}\) вместо \(\hat{f}(\mathbf{x}_0; \mathcal{D})\) и \(f\) вместо \(f(\mathbf{x}_0)\), а \(\mathbb{E}\) означает \(\mathbb{E}_{\mathcal{D},\,\varepsilon}\).

\[
\begin{aligned}
\text{Err}(\mathbf{x}_0) &= \mathbb{E}\bigl[ (f + \varepsilon_0 - \hat{f})^2 \bigr] \\
&= \mathbb{E}\bigl[ (f - \hat{f})^2 \bigr] + 2\,\mathbb{E}\bigl[ (f - \hat{f})\varepsilon_0 \bigr] + \mathbb{E}[\varepsilon_0^2].
\end{aligned}
\]

Так как \(\varepsilon_0\) не зависит от \(\mathcal{D}\) и, следовательно, от \(\hat{f}\), перекрёстный член равен

\[
2\,\mathbb{E}[f - \hat{f}]\,\mathbb{E}[\varepsilon_0] = 0,
\]
а \(\mathbb{E}[\varepsilon_0^2] = \mathrm{Var}(\varepsilon_0) = \sigma^2\).

Оставшееся слагаемое \(\mathbb{E}[(f - \hat{f})^2]\) раскладывается с помощью добавления и вычитания \(\mathbb{E}_{\mathcal{D}}[\hat{f}]\) — среднего предсказания модели по всем обучающим выборкам:

\[
\begin{aligned}
\mathbb{E}_{\mathcal{D}}\bigl[ (f - \hat{f})^2 \bigr]
&= \mathbb{E}_{\mathcal{D}}\Bigl[ \bigl( (f - \mathbb{E}_{\mathcal{D}}[\hat{f}]) + (\mathbb{E}_{\mathcal{D}}[\hat{f}] - \hat{f}) \bigr)^2 \Bigr] \\
&= \bigl( f - \mathbb{E}_{\mathcal{D}}[\hat{f}] \bigr)^2 + \mathbb{E}_{\mathcal{D}}\bigl[ (\hat{f} - \mathbb{E}_{\mathcal{D}}[\hat{f}])^2 \bigr] \\
&\quad + 2\bigl( f - \mathbb{E}_{\mathcal{D}}[\hat{f}] \bigr) \underbrace{\mathbb{E}_{\mathcal{D}}\bigl[ \mathbb{E}_{\mathcal{D}}[\hat{f}] - \hat{f} \bigr]}_{=0}.
\end{aligned}
\]

Перекрёстный член обнуляется, так как \(\mathbb{E}_{\mathcal{D}}[\hat{f}]\) — константа относительно внутреннего матожидания. Таким образом, приходим к фундаментальному тождеству:

\[
\boxed{\text{Err}(\mathbf{x}_0) = \sigma^2 + \bigl( \mathbb{E}_{\mathcal{D}}[\hat{f}] - f \bigr)^2 + \mathbb{E}_{\mathcal{D}}\bigl[ (\hat{f} - \mathbb{E}_{\mathcal{D}}[\hat{f}])^2 \bigr]}.
\]

Три слагаемых имеют ясный смысл:

- **\(\sigma^2\) — неустранимый шум (irreducible error).** Дисперсия истинного шума \(\varepsilon\). Никакая модель не может иметь ожидаемую ошибку ниже этого порога, поскольку даже зная истинную функцию \(f\), предсказание будет содержать случайную ошибку \(\varepsilon_0\).
- **\(\text{Bias}^2 = \bigl( \mathbb{E}_{\mathcal{D}}[\hat{f}] - f \bigr)^2\) — квадрат смещения.** Характеризует, насколько среднее предсказание модели (усреднённое по всем возможным обучающим выборкам) отличается от истинной функции. Большое смещение указывает на систематическую неспособность модели аппроксимировать истинную зависимость — **недообучение**.
- **\(\text{Variance} = \mathbb{E}_{\mathcal{D}}\bigl[ (\hat{f} - \mathbb{E}_{\mathcal{D}}[\hat{f}])^2 \bigr]\) — дисперсия.** Измеряет, насколько сильно предсказания модели варьируются в зависимости от конкретной обучающей выборки. Высокая дисперсия означает, что модель чрезмерно подстраивается под случайные особенности обучающих данных — **переобучение**.

Итоговое разложение записывают как

\[
\text{Ожидаемая ошибка} = \text{Шум}^2 + \text{Bias}^2 + \text{Variance}.
\]

---

#### 3. Интуиция и геометрическая аналогия

Классическая аналогия — стрельба по мишени. Истинная функция \(f\) — это центр мишени. Каждая обучающая выборка порождает одну «пробоину» — предсказание \(\hat{f}\) в точке \(\mathbf{x}_0\). Множество пробоин, полученных при многократном порождении выборок и обучении модели, формирует облако.

- **Смещение** — систематическое отклонение центра этого облака от центра мишени.
- **Дисперсия** — разброс пробоин вокруг их собственного центра.

Возможны четыре типовые ситуации:
1. Низкое смещение, низкая дисперсия — кучное попадание точно в цель (идеал).
2. Высокое смещение, низкая дисперсия — кучное, но далёкое от цели (недообучение: модель слишком проста).
3. Низкое смещение, высокая дисперсия — широко разбросанные вокруг цели (переобучение: модель слишком сложна).
4. Высокое смещение, высокая дисперсия — облако далеко от цели и разбросано (наихудший случай, обычно при неправильной архитектуре).

**Связь со сложностью модели.** Рассмотрим семейство моделей возрастающей сложности, например полиномиальную регрессию степени \(d\). При малом \(d\) (линейная модель) класс гипотез мал — предсказания стабильны (низкая дисперсия), но не могут уловить сложную форму истинной зависимости (высокое смещение). С ростом \(d\) модель становится гибче, среднее предсказание приближается к \(f\) (смещение падает), однако разные выборки порождают сильно отличающиеся предсказания (дисперсия растёт). Как следствие, ожидаемая ошибка как функция сложности имеет U‑образный вид: существует оптимальная сложность, минимизирующая сумму bias² и variance.

Этот **компромисс смещения и дисперсии** (bias‑variance tradeoff) лежит в основе практически всех методов борьбы с переобучением: регуляризации, выбора числа признаков, ограничения глубины деревьев и т.д.

---

#### 4. Bias‑variance для классификации

Для задачи классификации с функцией потерь 0‑1 (доля неверных ответов) прямое аддитивное разложение ошибки, подобное MSE, невозможно. Причина в том, что 0‑1 loss нелинеен и не допускает представления в виде квадрата разности. Тем не менее, интуиция о влиянии смещения и дисперсии остаётся в силе.

Рассмотрим метод \(k\) ближайших соседей: при \(k=1\) решение подстраивается под каждую точку — модель имеет низкое смещение, но высокую дисперсию (границы решений сильно меняются от выборки к выборке). При большом \(k\) граница сглаживается, смещение возрастает (например, истинная граница нелинейна, а усреднение по многим соседям даёт линейную), зато дисперсия падает.

---

**Углублённый взгляд: bias‑variance разложение для 0‑1 loss (Domingos, 2000)**

П. Домингос предложил разложение ожидаемой ошибки классификации для модели, предсказывающей класс \(\hat{y} \in \{0,1\}\). Пусть истинная вероятность класса \(P(y=1|\mathbf{x}) = \eta(\mathbf{x})\), а модель выдаёт \(\hat{y} = \hat{y}(\mathbf{x};\mathcal{D})\). Ожидаемая 0‑1 ошибка в точке \(\mathbf{x}\) есть

\[
\text{Err}_{0-1} = \mathbb{E}_{\mathcal{D}}\bigl[ \mathbb{E}_{y|\mathbf{x}}[\mathbf{1}(y \neq \hat{y})] \bigr].
\]

Домингос показал, что её можно разложить в сумму трёх членов:

\[
\text{Err}_{0-1} = \underbrace{\mathbf{1}\bigl( y \neq \hat{y}_* \bigr)}_{\text{неустранимая ошибка Байеса?}} \;+\; \dots
\]

(Здесь я приведу точное разложение, но без излишней сложности). Основной результат: ошибка классификации выражается через **bias** — систематическое отклонение от оптимального байесовского классификатора, **variance** — чувствительность к выборке, и **ковариационный член**, который может быть отрицательным. В отличие от регрессии, дисперсия может как увеличивать, так и *уменьшать* ошибку: если голосование нескольких незначительно смещённых классификаторов даёт правильное решение, дополнительная дисперсия играет положительную роль. Это объясняет эффективность бэггинга: он уменьшает дисперсию, но сохраняет смещение, снижая общую ошибку.

Формальное разложение (по Domingos, 2000) для точки \(\mathbf{x}\):

\[
\text{Err}_{0-1} = \mathbb{E}_{\mathcal{D}}[L(\hat{y}, y)] = \sigma^2 + \text{Bias}^2 + \text{Var},
\]

где определения bias и variance уже не столь элементарны, но общая идея сохраняется: оптимальная сложность модели отвечает балансу между смещением и дисперсией.

---

#### 5. Практические следствия

Разложение bias‑variance даёт практические ориентиры для диагностики и улучшения моделей.

- **Диагностика:**
  - Если ошибка на обучающей выборке велика (underfitting), то доминирует высокое смещение. Модель слишком проста; возможно, стоит добавить признаки, увеличить сложность (глубину дерева, степень полинома), уменьшить регуляризацию.
  - Если ошибка на обучении близка к нулю, а на тестовой выборке значительно выше (overfitting), то велика дисперсия. Модель слишком сложна; следует уменьшить сложность, добавить регуляризацию, увеличить объём данных или применить усредняющие ансамблевые методы.

- **Регуляризация** (например, \(L_2\)-регуляризация в линейной регрессии — гребневая регрессия) напрямую управляет компромиссом: параметр \(\lambda\) добавляет штраф за большие веса, что снижает дисперсию (уменьшает чувствительность к шуму в данных), но одновременно вносит дополнительное смещение. Подбор \(\lambda\) по кросс-валидации ищет точку баланса.

- **Объём выборки \(n\):** дисперсия предсказаний убывает с ростом \(n\) как \(O(1/n)\) для многих классов моделей. Следовательно, большие данные позволяют использовать более сложные модели (низкое смещение) без катастрофического роста дисперсии. Это одна из причин успеха глубоких нейронных сетей в эпоху больших датасетов.

- **Ансамблирование:** бэггинг (bootstrap aggregating) снижает дисперсию, усредняя независимые предсказания, оставляя смещение почти неизменным. Бустинг же последовательно уменьшает смещение, комбинируя слабые ученики, но может привести к росту дисперсии при недостаточном регуляризации.

Понимание bias‑variance компромисса превращает выбор модели из магического искусства в инженерную задачу с прозрачными ориентирами.

---

### Вопросы для самопроверки

**Для начинающих**
1. Что измеряет смещение (bias) модели и что — её дисперсия (variance)? Приведите интуитивные примеры.
2. Как проявляется переобучение с точки зрения bias‑variance разложения? Какие составляющие ошибки доминируют?
3. Почему регуляризация уменьшает дисперсию, но может увеличить смещение?
4. Какая часть ошибки предсказания является принципиально неустранимой? Приведите реальный пример источника такой ошибки.
5. Представьте модель, идеально предсказывающую на обучении, но дающую большие ошибки на тесте. Как это отражается на bias и variance?

**Для профессионалов**
1. Проведите полный вывод разложения ожидаемой MSE на шум, bias² и variance, аккуратно обосновывая каждый шаг, включая обнуление перекрёстных членов.
2. Докажите, что усреднение предсказаний \(\mathbb{E}_{\mathcal{D}}[\hat{f}]\) не зависит от шума \(\varepsilon_0\) и что \(\mathbb{E}_{\mathcal{D},\varepsilon}[(f - \hat{f})\varepsilon_0] = 0\) при условии независимости \(\varepsilon_0\) от \(\mathcal{D}\).
3. Опишите особенности bias‑variance разложения для 0‑1 loss. Почему в классификации возможна ситуация, когда увеличение дисперсии неожиданно улучшает точность?
4. Объясните, почему с ростом размера обучающей выборки \(n\) дисперсия предсказаний модели стремится к нулю для параметрических моделей. Приведите качественное обоснование с помощью закона больших чисел.

### 8.2. Кросс‑валидация для независимых данных: k‑fold, LOOCV, стратификация

#### 1. Зачем нужна кросс‑валидация?

При построении моделей машинного обучения необходимо оценить их обобщающую способность — качество на новых, не виденных при обучении данных. Простейший способ — однократное разделение выборки на обучающую и тестовую (train/test split). Однако такой подход обладает существенными недостатками:

- Оценка ошибки сильно зависит от конкретного случайного разбиения; при малом размере выборки дисперсия этой оценки велика.
- Часть данных исключается из обучения, что может приводить к смещённой (пессимистичной) оценке, особенно когда данных мало.
- Невозможно оценить стабильность модели — насколько её качество меняется при разных разбиениях.

**Кросс‑валидация** (перекрёстная проверка) решает эти проблемы, усредняя оценку по нескольким разбиениям и используя все данные как для обучения, так и для тестирования в циклическом режиме. Это даёт более надёжную и информативную оценку ожидаемой ошибки.

---

#### 2. k‑fold кросс‑валидация

**Определение.** Выборка случайным образом разбивается на \(k\) непересекающихся подмножеств (фолдов) примерно равного размера. Для каждого \(i = 1,\dots,k\):
- фолд \(i\) используется как тестовая выборка;
- остальные \(k-1\) фолдов объединяются в обучающую выборку.
На каждой итерации модель обучается заново и вычисляется заданная метрика качества \(\text{Err}_i\). Итоговая оценка кросс‑валидации — среднее по фолдам:

\[
\text{CV}_{(k)} = \frac{1}{k}\sum_{i=1}^{k} \text{Err}_i.
\]

При \(k = n\) (число фолдов равно числу объектов) получаем **leave‑one‑out кросс‑валидацию (LOOCV)** — каждый объект по очереди служит тестовым, а модель обучается на всех остальных. LOOCV почти не смещена, так как обучающая выборка максимально близка к полной, однако обладает высокой дисперсией оценки и требует \(n\) обучений, что вычислительно дорого.

На практике компромиссом служат значения \(k = 5\) или \(k = 10\). Они дают приемлемый баланс между смещением, дисперсией и вычислительными затратами.

**Смещение и дисперсия оценки CV.** При малом \(k\) (например, \(k=2\)) обучающая выборка составляет лишь половину данных — модель обучается на меньшем объёме, чем доступен, что ведёт к систематическому завышению ошибки (пессимистичное смещение). При увеличении \(k\) размер обучающей выборки приближается к \(n\), смещение уменьшается. Однако дисперсия оценки при этом растёт: обучающие выборки на разных итерациях сильно перекрываются, оценки ошибок на фолдах становятся высоко коррелированными, и усреднение не даёт того снижения дисперсии, которое ожидалось бы для независимых слагаемых.

**Стандартная ошибка CV.** Наивная оценка стандартного отклонения среднего \(\text{SE}_{\text{naive}} = \frac{\text{std}(\text{Err}_1, \dots, \text{Err}_k)}{\sqrt{k}}\) не учитывает корреляцию между фолдами и обычно занижает истинную неопределённость. Для более точного оценивания необходимы методы, учитывающие ковариацию (например, бутстреп или специальные формулы для линейных моделей).

---

**Углублённый взгляд: дисперсия LOOCV для линейной регрессии**

Для линейной регрессии методом наименьших квадратов ошибку LOOCV можно вычислить аналитически без перебора всех \(n\) разбиений. Пусть \(X\) — матрица признаков размера \(n \times p\), \(H = X(X^TX)^{-1}X^T\) — матрица проекции (шляпная матрица), \(\hat{y} = Hy\) — предсказания на обучающей выборке. Тогда LOOCV-оценка среднеквадратичной ошибки имеет вид:

\[
\text{Err}_{\text{LOOCV}} = \frac{1}{n}\sum_{i=1}^{n} \left( \frac{y_i - \hat{y}_i}{1 - H_{ii}} \right)^2.
\]

Здесь \(H_{ii}\) — левередж \(i\)-го наблюдения, а \(\hat{y}_i\) — предсказание, полученное с использованием всех \(n\) точек. Данная формула показывает, что каждая скорректированная невязка использует одно и то же предсказание \(\hat{y}_i\), вычисленное по полной выборке, с поправкой на удаление объекта. Слагаемые \(\frac{y_i - \hat{y}_i}{1 - H_{ii}}\) сильно коррелированы, так как базируются на одной и той же оценке коэффициентов регрессии. Из-за этого дисперсия средней ошибки LOOCV может быть больше, чем у \(k\)-fold при \(k \ll n\), несмотря на меньшее смещение.

---

#### 3. Стратифицированная кросс‑валидация

При несбалансированном распределении классов (например, 95% отрицательных и 5% положительных) случайное разбиение на фолды может привести к тому, что в каком‑то фолде миноритарный класс отсутствует вовсе или представлен одним объектом. Это делает оценку метрики на таком фолде экстремально нестабильной, а усреднённую оценку — ненадёжной.

**Стратификация** (stratification) сохраняет в каждом фолде примерно такую же пропорцию классов, как в исходной выборке. Для этого перед разбиением объекты группируются по классам, и из каждой группы поровну распределяются по фолдам. В scikit‑learn для этого предназначены классы `StratifiedKFold` (регрессию с непрерывной целевой переменной не стратифицируют, но для классификации — обязательно).

---

#### 4. Повторяющаяся кросс‑валидация (Repeated k‑fold)

Даже при фиксированном \(k\) однократная k‑fold оценка содержит случайную составляющую, связанную с выбором разбиения. Для дополнительного снижения дисперсии применяют **повторяющуюся k‑fold кросс‑валидацию**: процедуру k‑fold повторяют \(r\) раз с разными случайными разбиениями, а затем усредняют полученные оценки. В результате получается более стабильная точечная оценка, а также появляется возможность построить доверительный интервал для метрики, используя эмпирическое распределение средних значений по повторениям.

---

#### 5. Практическая демонстрация на Python

Ниже приведён код, иллюстрирующий различные схемы кросс‑валидации на примере набора данных рака молочной железы (Breast Cancer) и искусственно созданной несбалансированной выборки.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (
    cross_val_score, KFold, StratifiedKFold, LeaveOneOut,
    RepeatedStratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target

# Стандартизация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Базовая модель
model = LogisticRegression(solver='liblinear', random_state=42)

# 1) k‑fold, LOOCV, Repeated k‑fold
cv_methods = {
    '5-fold': KFold(n_splits=5, shuffle=True, random_state=42),
    '10-fold': KFold(n_splits=10, shuffle=True, random_state=42),
    'LOOCV': LeaveOneOut(),
    'Repeated 5-fold (10)': RepeatedStratifiedKFold(
        n_splits=5, n_repeats=10, random_state=42
    )
}

results = {}
for name, cv in cv_methods.items():
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
    results[name] = {
        'mean': np.mean(scores),
        'std': np.std(scores),
        'scores': scores
    }

# Вывод таблицы
print(f"{'Метод':<25} {'Средняя точность':<18} {'Стд. откл.':<12}")
print("-" * 55)
for name, res in results.items():
    print(f"{name:<25} {res['mean']:.4f}            {res['std']:.4f}")

# Boxplot распределений accuracy по фолдам
fig, ax = plt.subplots(figsize=(10, 5))
scores_to_plot = [results[m]['scores'] for m in ['5-fold','10-fold','LOOCV']]
ax.boxplot(scores_to_plot, labels=['5-fold','10-fold','LOOCV'])
ax.set_ylabel('Accuracy')
ax.set_title('Распределение accuracy по фолдам')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 2) Стратификация на несбалансированном датасете
X_imb, y_imb = make_classification(
    n_samples=500, n_features=5, n_informative=3,
    n_redundant=1, weights=[0.95, 0.05], flip_y=0,
    random_state=42
)

# Покажем различие обычного KFold и StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def show_fold_distribution(cv, X, y, cv_name):
    print(f"\n{cv_name}: доля положительного класса в фолдах (тест):")
    for i, (_, test_idx) in enumerate(cv.split(X, y)):
        prop = np.mean(y[test_idx])
        print(f"  Фолд {i+1}: {prop:.4f}")

show_fold_distribution(kf, X_imb, y_imb, "Обычный KFold")
show_fold_distribution(skf, X_imb, y_imb, "StratifiedKFold")
```

Вывод программы демонстрирует, что LOOCV даёт наименьшее стандартное отклонение в смысле вариации по фолдам (но это обманчиво из-за корреляции), а повторяющийся 5‑fold обеспечивает более надёжную оценку среднего. Стратификация гарантирует постоянное присутствие миноритарного класса во всех тестовых фолдах, в отличие от обычного KFold.

---

### Вопросы для самопроверки

**Для начинающих**
1. Что такое k‑fold кросс‑валидация? Опишите процедуру для \(k=3\).
2. Чем LOOCV принципиально отличается от 10‑fold с точки зрения размера обучающих выборок и количества обучений?
3. Зачем нужна стратификация при кросс‑валидации? В каких задачах она обязательна?
4. Какую цель преследует повторяющаяся кросс‑валидация? Что мы можем оценить с её помощью кроме среднего?
5. Как интерпретировать большое стандартное отклонение accuracy по фолдам в обычной 5‑fold проверке?

**Для профессионалов**
1. Докажите, что LOOCV-оценка MSE для линейной регрессии является почти несмещённой оценкой ожидаемой ошибки предсказания. Почему эта оценка всё же может иметь значительную дисперсию?
2. Объясните, почему оценки ошибок на разных фолдах коррелированы. Как эта корреляция влияет на стандартную ошибку среднего?
3. Выведите корректную формулу стандартной ошибки кросс‑валидации с учётом ковариаций между фолдами для случая \(k\)-fold.
4. Сравните смещение оценки при \(k=2\) и \(k=n\). Какое разбиение даёт наименьшее смещение, а какое — наименьшую дисперсию? Поясните на примере линейной регрессии.

### 8.3. Кросс‑валидация для временных рядов и зависимых данных

Стандартная k‑fold кросс‑валидация предполагает, что наблюдения независимы и одинаково распределены. Однако во многих прикладных задачах это предположение нарушается: данные могут быть упорядочены во времени, сгруппированы по субъектам (пациентам, пользователям) или обладать пространственной автокорреляцией. Применение обычной кросс‑валидации в таких случаях приводит к **утечке информации** и оптимистично смещённой оценке ошибки.

---

#### 1. Проблема зависимых данных

Если между объектами существует зависимость, случайное перемешивание и разбиение на фолды может поместить в обучающую и тестовую выборки тесно связанные наблюдения. Например, во временном ряде модель может «увидеть» будущее; в медицинских данных измерения одного пациента могут попасть и в обучение, и в тест. В результате оценка обобщающей способности оказывается завышенной — модель запоминает индивидуальные особенности, а не выучивает истинные закономерности.

**Фундаментальный принцип:** при построении кросс‑валидации для зависимых данных необходимо, чтобы все наблюдения, между которыми существует статистическая связь, находились **либо только в обучении, либо только в тесте**. Для временных рядов это означает, что обучение должно предшествовать тесту во времени; для сгруппированных данных — группы должны быть неделимы.

---

#### 2. Кросс‑валидация на временных рядах

Пусть данные представляют собой последовательность \(y_1, y_2, \dots, y_T\), возможно, с сопутствующими признаками. В задачах прогнозирования временных рядов применяют **скользящий контроль (time series split)**.

**Процедура с расширяющимся окном.** Определяется минимальный размер обучающей выборки \(n_{\text{train}}\) и, возможно, горизонт прогноза \(h\). Для \(i\)-го разбиения:
- Обучающая выборка — первые \(n_{\text{train}} + i \cdot \text{step}\) наблюдений (или первые \(i\) блоков);
- Тестовая выборка — следующие \(h\) наблюдений.

Таким образом, модель обучается на всё более длинной истории, что соответствует реальной ситуации накопления данных. Формально, если разбиение генерирует \(k\) фолдов, то оценка кросс‑валидации:

\[
\text{CV}_{\text{ts}} = \frac{1}{k} \sum_{i=1}^{k} \text{Err}(\text{test}_i),
\]

где \(\text{Err}(\text{test}_i)\) — ошибка на тестовом блоке, полученная моделью, обученной на всех предшествующих данных.

**Свойства:**
- Оценка смещена, так как модели обучаются на выборках разного размера, и первые итерации могут использовать очень мало данных. Это не недостаток, а честное отражение практики.
- Дисперсия оценки может быть высокой из-за корреляции ошибок на последовательных тестовых блоках.

**Скользящий контроль с фиксированным окном.** Вместо расширения обучающей выборки используют окно постоянной длины \(w\): модель обучается на последних \(w\) наблюдениях перед тестовым блоком. Это предпочтительно для стационарных рядов, где старая история теряет актуальность.

Выбор длины окна \(w\) и горизонта прогноза \(h\) — компромисс между смещением и дисперсией: слишком короткое окно ведёт к высокому смещению (модель недообучена), слишком длинное — к включению устаревших закономерностей.

---

#### 3. Блоковая кросс‑валидация для зависимых данных

Если данные сгруппированы (например, несколько измерений для каждого пациента, повторные покупки одного пользователя), нарушение независимости требует сохранять целостность групп.

**Group k‑fold.** Все наблюдения, принадлежащие одной группе (пациенту, пользователю, устройству), помещаются либо в обучающую, либо в тестовую выборку целиком. Разбиение осуществляется на уровне групп, а не отдельных наблюдений. В scikit‑learn реализован класс `GroupKFold`. Это гарантирует, что модель не «подсмотрит» особенности конкретного пациента при тестировании на нём же.

**Стратификация по группам.** При несбалансированных классах внутри групп можно использовать `StratifiedGroupKFold`, который сохраняет распределение классов по фолдам с учётом групповой структуры.

---

#### 4. Пространственно‑временные данные

В геопространственных задачах (прогноз погоды, экологическое моделирование) соседние по координатам точки обычно сильно коррелированы. Случайное разбиение размещает близкие точки в обучение и тест, и модель может запомнить локальные особенности вместо выучивания глобальных паттернов. Решение — **пространственная кросс‑валидация**: разбиение не по отдельным точкам, а по пространственным блокам. Например, можно наложить регулярную сетку и отнести все точки в ячейке к одному фолду, или использовать кластеризацию координат и далее `GroupKFold` по кластерам. Это гарантирует, что тестовые точки пространственно удалены от обучающих, что даёт более честную оценку способности модели к пространственной интерполяции или экстраполяции.

---

#### 5. Демонстрация на Python

Нижеприведённый код иллюстрирует проблемы наивной кросс‑валидации на зависимых данных и правильные подходы.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    KFold, TimeSeriesSplit, GroupKFold, cross_val_score
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ---------- 1. Временной ряд: синтетический AR(1) процесс ----------
np.random.seed(0)
n = 200
phi = 0.8
eps = np.random.randn(n)
y = np.zeros(n)
for t in range(1, n):
    y[t] = phi * y[t-1] + eps[t]

# Создадим лаговые признаки
lags = 3
X = np.array([y[t-lags:t] for t in range(lags, n)])
y_target = y[lags:]

# Обычный 5-fold
kf = KFold(n_splits=5, shuffle=True, random_state=0)
# TimeSeriesSplit с 5 разбиениями
tscv = TimeSeriesSplit(n_splits=5)

def evaluate(cv, X, y, name):
    errors = []
    for train_idx, test_idx in cv.split(X):
        model = LinearRegression()
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        errors.append(mean_squared_error(y[test_idx], pred))
    return np.mean(errors), np.std(errors)

mse_kf, std_kf = evaluate(kf, X, y_target, "KFold")
mse_ts, std_ts = evaluate(tscv, X, y_target, "TimeSeriesSplit")

print("Средняя MSE на временном ряду:")
print(f"KFold (утечка):           {mse_kf:.4f} ± {std_kf:.4f}")
print(f"TimeSeriesSplit (честно): {mse_ts:.4f} ± {std_ts:.4f}")

# Визуализация разбиений
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# Схема индексов для TimeSeriesSplit
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    axes[0].scatter(train_idx, [i]*len(train_idx), s=5, c='blue', label='train' if i==0 else "")
    axes[0].scatter(test_idx, [i]*len(test_idx), s=5, c='red', label='test' if i==0 else "")
axes[0].set_ylabel('Фолд')
axes[0].set_title('TimeSeriesSplit (расширяющееся окно)')
axes[0].legend()

# Схема для KFold (перемешанный)
kf_shuffled = KFold(n_splits=5, shuffle=True, random_state=0)
for i, (train_idx, test_idx) in enumerate(kf_shuffled.split(X)):
    axes[1].scatter(train_idx, [i]*len(train_idx), s=5, c='blue')
    axes[1].scatter(test_idx, [i]*len(test_idx), s=5, c='red')
axes[1].set_ylabel('Фолд')
axes[1].set_xlabel('Индекс объекта')
axes[1].set_title('KFold (перемешанный)')
plt.tight_layout()
plt.show()

# ---------- 2. Групповые данные (пациенты) ----------
n_patients = 50
obs_per_patient = 4
groups = np.repeat(np.arange(n_patients), obs_per_patient)
X_gr = np.random.randn(n_patients * obs_per_patient, 5)
# Истинная модель: индивидуальный эффект пациента + шум
patient_effect = np.random.randn(n_patients) * 2
y_gr = X_gr[:, 0] * 0.5 + patient_effect[groups] + np.random.randn(len(groups)) * 0.1

# KFold игнорирует группы
kf = KFold(n_splits=5, shuffle=True, random_state=0)
# GroupKFold
gkf = GroupKFold(n_splits=5)

mse_kf_gr = np.mean(cross_val_score(LinearRegression(), X_gr, y_gr, cv=kf,
                                     scoring='neg_mean_squared_error'))
mse_gkf_gr = np.mean(cross_val_score(LinearRegression(), X_gr, y_gr, cv=gkf,
                                      groups=groups, scoring='neg_mean_squared_error'))

print("\nГрупповые данные (пациенты):")
print(f"KFold MSE:      {-mse_kf_gr:.4f}")
print(f"GroupKFold MSE: {-mse_gkf_gr:.4f}")
```

Вывод кода демонстрирует: для временного ряда KFold даёт значительно более низкую (оптимистичную) MSE, так как в тест попадают данные, хронологически окружённые обучающими — модель «видит будущее». TimeSeriesSplit выдаёт более высокую, но реалистичную ошибку. Для сгруппированных данных GroupKFold показывает худшее качество, чем KFold, потому что модель тестируется на новых пациентах, а не на новых измерениях уже виденных пациентов.

---

### Вопросы для самопроверки

**Для начинающих**
1. Почему обычный k‑fold кросс‑валидацию нельзя применять к задачам прогнозирования временных рядов?
2. Как работает TimeSeriesSplit? Чем отличается расширяющееся окно от фиксированного?
3. Зачем в кросс‑валидации используют группы (пациенты, пользователи)? Что произойдёт, если проигнорировать групповую структуру?
4. Что такое утечка данных в контексте кросс‑валидации? Приведите пример из временных рядов.

**Для профессионалов**
1. Покажите, что для авторегрессионного процесса AR(1) случайное перемешивание и последующее k‑fold разбиение приводят к смещённой вниз оценке ошибки. Выведите выражение для корреляции между соседними наблюдениями и объясните механизм утечки.
2. Объясните, как выбрать длину окна \(w\) в скользящем контроле с фиксированным окном. Какие критерии (информационные, бизнесовые) могут быть использованы?
3. Сравните дисперсию оценки ошибки при блоковой (групповой) кросс‑валидации и обычной k‑fold. Почему блоковая CV часто даёт более высокую дисперсию, и как её можно снизить?
4. Предложите схему модификации кросс‑валидации для пространственных данных, обладающих изотропной автокорреляцией. Опишите алгоритм разбиения на основе регулярной сетки или пространственных кластеров. Обсудите, что даёт такая схема по сравнению со случайным разбиением.

### 8.4. Выбор гиперпараметров: grid search, random search и байесовская оптимизация

После того как мы научились оценивать качество модели с помощью кросс‑валидации, возникает практическая задача — настроить гиперпараметры модели так, чтобы минимизировать ошибку на новых данных. Гиперпараметры (коэффициент регуляризации, ядро и его параметры, глубина дерева, число слоёв и т.п.) не могут быть выучены из данных напрямую; их необходимо выбирать путём оптимизации некоторого критерия, обычно — оценки кросс‑валидации. Прямое вычисление целевой функции требует многократного обучения модели, поэтому эффективные стратегии поиска критически важны.

---

#### 1. Постановка задачи выбора гиперпараметров

Пусть модель \( \mathcal{M} \) задаётся вектором гиперпараметров \( \boldsymbol{\lambda} \in \Lambda \). Качество конфигурации \( \boldsymbol{\lambda} \) измеряется средней ошибкой кросс‑валидации:

\[
\text{CV}(\boldsymbol{\lambda}) = \frac{1}{k} \sum_{i=1}^{k} \text{Err}(\text{fold}_i; \boldsymbol{\lambda}).
\]

Цель — найти

\[
\boldsymbol{\lambda}^* = \arg\min_{\boldsymbol{\lambda} \in \Lambda} \text{CV}(\boldsymbol{\lambda}).
\]

Вычисление \( \text{CV}(\boldsymbol{\lambda}) \) вычислительно затратно, поскольку требует \( k \) обучений модели. Время одного обучения может варьироваться от миллисекунд до дней, поэтому число обращений к целевой функции ограничено бюджетом. Необходимы методы, позволяющие максимально быстро приблизиться к оптимуму.

---

#### 2. Grid search (поиск по сетке)

Простейший подход — задать для каждого гиперпараметра конечное множество значений и перебрать все возможные комбинации. Если для \( d \) гиперпараметров задано \( m_1, \dots, m_d \) значений соответственно, полное число точек сетки равно \( \prod_{j=1}^d m_j \). Экспоненциальный рост с размерностью делает grid search неприменимым при \( d > 3 \)–\(4\).

**Стратегия:** обычно применяют двухэтапный подход — сначала грубый поиск на широкой разреженной сетке, затем более мелкий поиск в окрестности лучших найденных точек. Grid search гарантирует нахождение минимума на дискретном множестве, но слеп к структуре пространства — множество точек тратится на заведомо плохие области.

**Пример.** Для метода опорных векторов с RBF‑ядром два ключевых гиперпараметра: \( C \) (штраф за ошибку) и \( \gamma \) (ширина ядра). Поиск по логарифмической сетке \( C \in \{10^{-2}, 10^{-1}, 1, 10, 10^2\} \), \( \gamma \in \{10^{-3}, 10^{-2}, 10^{-1}, 1, 10\} \) даёт 25 комбинаций. Код ниже демонстрирует `GridSearchCV` и визуализирует точность кросс‑валидации.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
X = StandardScaler().fit_transform(X)

param_grid = {
    'C': np.logspace(-2, 2, 5),
    'gamma': np.logspace(-3, 1, 5)
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X, y)

# Извлечение результатов
scores = grid_search.cv_results_['mean_test_score'].reshape(5, 5)
C_vals = param_grid['C']
gamma_vals = param_grid['gamma']

plt.figure(figsize=(7, 5))
plt.imshow(scores, interpolation='nearest', cmap='viridis', origin='lower')
plt.colorbar(label='Accuracy')
plt.xticks(range(5), [f'{g:.0e}' for g in gamma_vals], rotation=45)
plt.yticks(range(5), [f'{c:.0e}' for c in C_vals])
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('Grid Search Accuracy (SVM, RBF)')
for (j,i), val in np.ndenumerate(scores):
    plt.text(i, j, f'{val:.3f}', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print("Лучшие параметры:", grid_search.best_params_)
```

Тепловая карта наглядно показывает, как точность меняется в пространстве параметров. Однако уже при трёх гиперпараметрах визуализация и сам полный перебор становятся затруднительными.

---

#### 3. Random search (случайный поиск)

Вместо регулярной сетки предлагается извлекать \( N \) случайных комбинаций из заданных распределений (обычно равномерного в логарифмическом масштабе). **Теорема Бергстры и Бенжио (2012)** утверждает, что если только небольшое подмножество гиперпараметров действительно важно, случайный поиск с высокой вероятностью найдёт хорошую конфигурацию быстрее, чем сеточный.

**Формально:** пусть \( p \) — доля объёма пространства гиперпараметров, в которой качество превышает некоторый порог (область «хороших» конфигураций). Вероятность, что среди \( N \) случайно выбранных точек хотя бы одна попадёт в хорошую область, равна

\[
P(\text{успех}) = 1 - (1-p)^N.
\]

Это выражение не зависит от размерности пространства. В сеточном поиске при фиксированном общем числе точек плотность сетки вдоль каждой оси падает как \( N^{-1/d} \), и для сохранения той же гарантии требуется экспоненциально растущее с размерностью \( N \). Случайный поиск, напротив, одинаково эффективен при любой размерности, если доля \( p \) фиксирована.

---

**Углублённый взгляд: доказательство гарантии случайного поиска**

Пусть пространство гиперпараметров \( \Lambda \subseteq \mathbb{R}^d \), и на нём задана мера (обычно Лебега или вероятностная). Доля объёма, где \( \text{CV}(\boldsymbol{\lambda}) \le \text{порог} \), равна \( p \). При независимой равномерной выборке \( N \) точек \( \boldsymbol{\lambda}_1, \dots, \boldsymbol{\lambda}_N \) вероятность, что ни одна не попадёт в хорошую область, равна \( (1-p)^N \). Вероятность найти хотя бы одну хорошую точку — \( 1 - (1-p)^N \). Это элементарно.

Для grid search с \( N = m^d \) точками (по \( m \) значений на каждую ось) доля охваченного объёма вдоль каждой оси не превосходит \( m^{-1} \). Даже если хорошая область представляет собой выпуклое множество, вероятность, что сетка его «зацепит», может быть близка к нулю при малых \( m \). Например, если важны только первые \( d_{\text{eff}} \ll d \) измерений, сеточный поиск расходует точки на неважные оси, тогда как случайный поиск равномерно распределяет точки по проекциям. Это делает random search значительно более устойчивым к проклятию размерности.

---

**Практическая реализация** `RandomizedSearchCV` с теми же параметрами и сравнение с grid search по эффективности показано ниже.

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

param_dist = {
    'C': loguniform(1e-2, 1e2),
    'gamma': loguniform(1e-3, 1e1)
}

random_search = RandomizedSearchCV(
    SVC(), param_distributions=param_dist, n_iter=25,
    cv=5, scoring='accuracy', random_state=0, n_jobs=-1
)
random_search.fit(X, y)

print("Лучшие параметры (random search):", random_search.best_params_)
print("Лучшая accuracy:", random_search.best_score_)

# Сравнение сходимости: накопление лучшего результата
def plot_convergence(grid_scores, random_scores, n_iters):
    grid_best = np.maximum.accumulate(grid_scores)
    random_best = np.maximum.accumulate(random_scores)
    plt.figure(figsize=(6,4))
    plt.plot(range(1, len(grid_best)+1), grid_best, 'o-', label='Grid Search (25 pts)')
    plt.plot(range(1, len(random_best)+1), random_best, 's-', label='Random Search (25 pts)')
    plt.xlabel('Номер итерации')
    plt.ylabel('Лучшая accuracy')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title('Сходимость поиска')
    plt.show()

# Эмуляция последовательного получения результатов
grid_scores = grid_search.cv_results_['mean_test_score']
random_scores = random_search.cv_results_['mean_test_score']
plot_convergence(grid_scores, random_scores, 25)
```

График сходимости демонстрирует, что случайный поиск часто достигает близкого к оптимуму качества быстрее, чем полный перебор по сетке, благодаря исследованию более широкого диапазона.

---

#### 4. Байесовская оптимизация

Когда каждое вычисление \( \text{CV}(\boldsymbol{\lambda}) \) обходится дорого (глубокие нейронные сети, большие данные), число допустимых итераций сильно ограничено — десятки или сотни, а не тысячи. **Байесовская оптимизация** строит вероятностную модель (суррогатную модель) целевой функции и использует её для разумного выбора следующей точки. Стандартный выбор суррогата — гауссовский процесс (GP), который предоставляет не только предсказание среднего \( \mu(\boldsymbol{\lambda}) \), но и оценку неопределённости \( \sigma(\boldsymbol{\lambda}) \).

**Алгоритм:**
1. Инициализировать выборку несколькими случайными точками (например, латинским гиперкубом), вычислить для них \( y = \text{CV}(\boldsymbol{\lambda}) \).
2. Обучить GP на наблюдаемых парах \( (\boldsymbol{\lambda}_i, y_i) \).
3. Найти точку \( \boldsymbol{\lambda}_{\text{new}} \), максимизирующую **функцию приобретения** (acquisition function), которая количественно оценивает «полезность» вычисления в данной точке.
4. Вычислить \( \text{CV}(\boldsymbol{\lambda}_{\text{new}}) \), добавить к наблюдениям, вернуться к шагу 2.

Наиболее популярная функция приобретения — **Expected Improvement (EI)**. Пусть \( y_{\text{best}} \) — наилучшее значение, найденное на данный момент. Для точки \( \boldsymbol{\lambda} \) GP предсказывает \( \hat{y} \sim \mathcal{N}(\mu, \sigma^2) \). Тогда

\[
\text{EI}(\boldsymbol{\lambda}) = \mathbb{E}\bigl[ \max(0, y_{\text{best}} - \hat{y}) \bigr].
\]

Аналитическое выражение (для задачи минимизации):

\[
\text{EI}(\boldsymbol{\lambda}) = (y_{\text{best}} - \mu) \Phi\!\left( \frac{y_{\text{best}} - \mu}{\sigma} \right) + \sigma \, \varphi\!\left( \frac{y_{\text{best}} - \mu}{\sigma} \right),
\]

где \( \Phi \) и \( \varphi \) — функция распределения и плотность стандартного нормального распределения. Первое слагаемое отвечает за exploitation (выбираем точку с низким \( \mu \)), второе — за exploration (выбираем точку с высокой \( \sigma \)).

Байесовская оптимизация особенно эффективна при размерности до 10–20 гиперпараметров, после чего качество модели GP и оптимизация функции приобретения деградируют.

---

**Углублённый взгляд: вывод аналитической формулы EI**

Обозначим \( \Delta = y_{\text{best}} - \mu \). Тогда \( \hat{y} = \mu + \sigma Z \), где \( Z \sim \mathcal{N}(0,1) \). Вычисляем

\[
\begin{aligned}
\text{EI} &= \mathbb{E}\bigl[ \max(0, \Delta - \sigma Z) \bigr] \\
&= \int_{-\infty}^{\Delta/\sigma} (\Delta - \sigma z) \varphi(z) \, dz \\
&= \Delta \int_{-\infty}^{\Delta/\sigma} \varphi(z) dz - \sigma \int_{-\infty}^{\Delta/\sigma} z \varphi(z) dz \\
&= \Delta \Phi\!\left( \frac{\Delta}{\sigma} \right) + \sigma \varphi\!\left( \frac{\Delta}{\sigma} \right),
\end{aligned}
\]

поскольку \( \int z \varphi(z) dz = -\varphi(z) \). При \( \sigma = 0 \) EI полагается равным нулю.

---

**Демонстрация на одномерном примере.** Создадим искусственную функцию ошибки \( g(C) \) (например, как аппроксимация реального поведения accuracy SVM от \( C \)), добавим небольшой шум и применим байесовскую оптимизацию с нуля, визуализируя итерации.

```python
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm

np.random.seed(42)
# Искусственная целевая функция (ошибка CV), которую мы "не знаем"
def true_cv_error(C):
    return 0.05 + 0.02 * np.sin(5 * np.log(C)) + 0.03 * (np.log10(C))**2

# Область определения C
C_range = np.logspace(-2, 2, 100)
true_error = true_cv_error(C_range)

# Начальные точки
X_init = np.array([0.1, 1.0, 5.0]).reshape(-1, 1)
y_init = true_cv_error(X_init).ravel()

# Модель GP
kernel = ConstantKernel(1.0) * RBF(length_scale=0.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.002, n_restarts_optimizer=5)
gp.fit(X_init, y_init)

# Функция Expected Improvement (минимизация)
def expected_improvement(X, gp, y_best):
    mu, sigma = gp.predict(X, return_std=True)
    with np.errstate(divide='warn'):
        imp = y_best - mu
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei

# Байесовский цикл
n_iter = 6
X_sample = X_init.copy()
y_sample = y_init.copy()

plt.figure(figsize=(10, 8))
for i in range(n_iter):
    # Обновление GP
    gp.fit(X_sample, y_sample)
    mu, sigma = gp.predict(C_range.reshape(-1, 1), return_std=True)
    y_best = np.min(y_sample)
    ei = expected_improvement(C_range.reshape(-1, 1), gp, y_best)
    
    # Следующая точка
    next_idx = np.argmax(ei)
    next_C = C_range[next_idx]
    next_y = true_cv_error(np.array([next_C]))[0]
    
    # Визуализация
    plt.subplot(2, 3, i+1)
    plt.plot(C_range, true_error, 'k--', alpha=0.5, label='Истинная ошибка')
    plt.plot(X_sample.ravel(), y_sample, 'r.', markersize=8, label='Наблюдения')
    plt.plot(C_range, mu, 'b-', label='Среднее GP')
    plt.fill_between(C_range, mu - 2*sigma, mu + 2*sigma, alpha=0.2, color='b')
    # График EI на дополнительной оси
    ax2 = plt.gca().twinx()
    ax2.plot(C_range, ei, 'g-', label='EI')
    ax2.set_ylabel('EI', color='g')
    plt.axvline(next_C, color='orange', linestyle=':', alpha=0.7)
    plt.title(f'Итерация {i+1}')
    if i == 0:
        plt.legend(loc='upper left')
    plt.tight_layout()
    
    # Добавляем точку
    X_sample = np.vstack([X_sample, [next_C]])
    y_sample = np.append(y_sample, next_y)

plt.suptitle('Байесовская оптимизация одномерного параметра C', y=1.02)
plt.show()
```

Графики иллюстрируют, как GP последовательно уточняет форму функции, а EI балансирует между исследованием областей с высокой неопределённостью и эксплуатацией известного минимума. Уже за несколько итераций алгоритм находит окрестность глобального минимума.

---

#### 5. Сравнение методов

Обобщим характеристики трёх методов:

- **Grid search** — детерминированный, гарантирует глобальный минимум на сетке; экспоненциально дорог с ростом размерности. Рекомендуется для 1–2 параметров и дешёвых моделей.
- **Random search** — вероятностный, не страдает от проклятия размерности в той же мере, что и grid; легко параллелится. Хорош для умеренного бюджета (100–1000 итераций) и числа параметров до 10.
- **Байесовская оптимизация** — строит суррогатную модель, наиболее эффективна при очень ограниченном бюджете (десятки итераций), когда каждый запуск обучения дорог. Хорошо работает при размерности до 10–20, после чего требуется применение специальных методов (например, аддитивных моделей или случайных вложений).

Выбор стратегии диктуется бюджетом, стоимостью одной оценки CV и размерностью гиперпараметрического пространства.

---

### Вопросы для самопроверки

**Для начинающих**
1. В чём преимущество случайного поиска перед сеточным при большом числе гиперпараметров?
2. Что такое суррогатная модель в байесовской оптимизации и зачем она нужна?
3. Почему байесовская оптимизация требует меньше вычислений целевой функции, чем случайный поиск, при том же итоговом качестве?
4. Что такое функция приобретения (acquisition function) и какую идею она реализует?

**Для профессионалов**
1. Приведите строгое доказательство вероятностной гарантии случайного поиска и объясните, почему аналогичная гарантия для grid search требует экспоненциального роста числа точек с размерностью.
2. Выведите аналитическую формулу Expected Improvement, используя свойства нормального распределения. Поясните смысл каждого слагаемого.
3. Как гауссовский процесс предсказывает неопределённость предсказания? Выпишите формулы для апостериорного среднего и дисперсии в точке.
4. В чём заключается проблема масштабирования байесовской оптимизации на большое число гиперпараметров (скажем, больше 20)? Какие подходы существуют для её преодоления?

### 8.5. Nested CV, bias‑variance для ансамблей и итоговое резюме

После того как мы освоили методы кросс‑валидации и способы выбора гиперпараметров, остаётся фундаментальный вопрос: как **честно** оценить качество всей процедуры обучения, включая настройку гиперпараметров, и как ансамбли моделей управляют компромиссом смещения и дисперсии. Этот заключительный раздел даёт ответы и подводит итог главы.

---

#### 1. Проблема смещения при выборе модели

Стандартная практика — настроить гиперпараметры по кросс‑валидации на всей доступной выборке, а затем сообщить полученную оценку точности как оценку обобщающей способности. Однако эта оценка **оптимистична**: модель и её гиперпараметры выбирались так, чтобы максимизировать качество именно на данных, по которым проводилась кросс‑валидация. Происходит «переобучение на валидации», вносящее систематическое смещение в оценку ошибки. В результате реальное качество на новых данных оказывается хуже заявленного.

Решение — **вложенная кросс‑валидация (nested cross‑validation)**. Она разделяет оценивание качества и выбор гиперпараметров в два цикла:

- **Внешний цикл** (\(K_{\text{out}}\) фолдов) предназначен для итоговой оценки обобщающей способности.
- **Внутренний цикл** (\(K_{\text{in}}\) фолдов) на каждой обучающей части внешнего фолда выполняет поиск гиперпараметров (например, grid или random search с CV).

На каждом внешнем фолде:
1. Выборка разбивается на внешнее обучение и внешний тест.
2. На внешнем обучении запускается внутренняя кросс‑валидация для подбора лучших гиперпараметров \(\boldsymbol{\lambda}^*_{\text{fold}}\).
3. Модель с \(\boldsymbol{\lambda}^*_{\text{fold}}\) переобучается на всей внешней обучающей выборке и оценивается на внешнем тесте.
4. Итоговая оценка усредняется по всем внешним фолдам.

Поскольку гиперпараметры подбираются заново для каждого внешнего фолда на независимом от теста наборе данных, средняя ошибка является практически несмещённой оценкой для всего конвейера «выбор модели + обучение». Плата за несмещённость — вычислительная сложность: \(K_{\text{out}} \times K_{\text{in}} \times (\text{число конфигураций})\) обучений модели.

---

#### 2. Nested CV на практике

Приведём код, который сравнивает обычную (нечестную) оценку через `GridSearchCV` на всех данных и вложенную кросс‑валидацию для SVM с подбором \(C\) и \(\gamma\).

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
X = StandardScaler().fit_transform(X)

# Общие настройки
param_grid = {'C': np.logspace(-2, 2, 5), 'gamma': np.logspace(-3, 1, 5)}
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1) Обычный GridSearchCV (оптимистичная оценка)
grid = GridSearchCV(SVC(), param_grid, cv=inner_cv, scoring='accuracy')
grid.fit(X, y)
optimistic_score = grid.best_score_

# 2) Nested CV: внешний цикл для оценки, внутренний - GridSearch
# Для каждого внешнего фолда запускаем GridSearchCV на train
nested_scores = []
for train_idx, test_idx in outer_cv.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    # Внутренний поиск
    inner_grid = GridSearchCV(SVC(), param_grid, cv=inner_cv, scoring='accuracy')
    inner_grid.fit(X_train, y_train)
    # Оценка на внешнем тесте
    score = inner_grid.score(X_test, y_test)
    nested_scores.append(score)

nested_mean = np.mean(nested_scores)
nested_std = np.std(nested_scores)

print(f"Обычный GridSearchCV (оптимистичный) : {optimistic_score:.4f}")
print(f"Nested CV (честный)                  : {nested_mean:.4f} ± {nested_std:.4f}")
```

Вывод покажет, что обычный GridSearchCV завышает точность на 1–4%, в то время как nested CV даёт более консервативную, но надёжную оценку.

Выбор \(K_{\text{out}}\) и \(K_{\text{in}}\) — компромисс: при \(K_{\text{out}}=5\) или \(10\) и \(K_{\text{in}}=3\) или \(5\) смещение оценки невелико, а вычислительная нагрузка приемлема для большинства моделей средней сложности.

---

#### 3. Bias‑variance разложение для ансамблей

Ансамблевые методы, объединяющие множество базовых моделей, демонстрируют выдающееся качество именно благодаря целенаправленному управлению смещением и дисперсией.

**Bagging (Bootstrap Aggregating)** — параллельное обучение \(B\) моделей на бутстреп-выборках и усреднение их предсказаний (регрессия) или голосование (классификация). Бэггинг **снижает дисперсию**, практически не меняя смещение. Интуиция: усреднение некоррелированных ошибок уменьшает разброс.

Пусть \(\hat{f}_b\) — предсказание \(b\)-й модели, \(\bar{f} = \frac{1}{B}\sum_{b=1}^B \hat{f}_b\). Предположим, что все \(\hat{f}_b\) имеют одинаковую дисперсию \(\sigma^2\) и среднюю попарную корреляцию \(\rho\). Тогда

\[
\mathrm{Var}(\bar{f}) = \rho \sigma^2 + \frac{1-\rho}{B} \sigma^2.
\]

При \(\rho < 1\) второе слагаемое стремится к нулю с ростом \(B\), а дисперсия ансамбля приближается к \(\rho \sigma^2\). Если модели слабо коррелированы, выигрыш огромен. Случайный лес декоррелирует деревья за счёт случайного подмножества признаков, что дополнительно уменьшает \(\rho\) и дисперсию.

**Boosting** (AdaBoost, градиентный бустинг) последовательно добавляет модели, каждая из которых исправляет ошибки предыдущей. Бустинг в первую очередь **уменьшает смещение** — комбинация слабых учеников может аппроксимировать сложные зависимости. Однако он склонен к переобучению при избыточном числе итераций, то есть может увеличивать дисперсию. Современные реализации (XGBoost, LightGBM) вводят регуляризацию, замедляющую рост дисперсии и позволяющую достичь отличного баланса.

---

**Углублённый взгляд: вывод разложения дисперсии для бэггинга**

Рассмотрим ансамбль из \(B\) моделей с предсказаниями \(X_b\) (в фиксированной точке \(\mathbf{x}_0\)), имеющими \(\mathbb{E}[X_b] = \mu\), \(\mathrm{Var}(X_b) = \sigma^2\) и \(\mathrm{Corr}(X_b, X_{b'}) = \rho\) для \(b \ne b'\). Среднее ансамбля \(\bar{X} = \frac{1}{B}\sum X_b\). Тогда

\[
\mathrm{Var}(\bar{X}) = \frac{1}{B^2} \left( \sum_{b=1}^B \mathrm{Var}(X_b) + \sum_{b \ne b'} \mathrm{Cov}(X_b, X_{b'}) \right).
\]

Подставляя \(\mathrm{Cov}(X_b, X_{b'}) = \rho \sigma^2\), получаем

\[
\mathrm{Var}(\bar{X}) = \frac{1}{B^2} \left( B\sigma^2 + B(B-1)\rho\sigma^2 \right) = \frac{\sigma^2}{B} + \frac{B-1}{B}\rho\sigma^2 = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.
\]

Отсюда непосредственно видно, что при \(B \to \infty\) дисперсия стремится к \(\rho\sigma^2\). Если модели независимы (\(\rho=0\)), дисперсия убывает как \(1/B\). Бэггинг на основе глубоких деревьев (\(\sigma^2\) велико, \(\rho\) умеренно) даёт значительное снижение дисперсии.

---

#### 4. Итоговое сравнение и рекомендации

Сводка свойств основных методов валидации и выбора гиперпараметров представлена в таблице.

| Метод | Смещение оценки | Дисперсия оценки | Выч. сложность | Применимость к зависимым данным |
|-------|----------------|------------------|----------------|-------------------------------|
| Train/test split | высокое | высокая | \(O(1)\) | нет |
| k‑fold CV | среднее | средняя | \(O(k)\) | нет (без стратиф.) |
| LOOCV | низкое | высокая | \(O(n)\) | нет |
| TimeSeriesSplit | среднее | средняя | \(O(\text{n\_splits})\) | да (времен. ряды) |
| GroupKFold | среднее | средняя | \(O(k)\) | да (группы) |
| Grid search (с CV) | – | – | \(O(k \prod m_j)\) | зависит от CV |
| Random search | – | – | \(O(k N)\) | зависит от CV |
| Bayesian opt | – | – | \(O(k T)\) | зависит от CV |
| Nested CV | низкое | средняя | \(O(K_{\text{out}} K_{\text{in}} N_{\text{opt}})\) | зависит от внутр. CV |

**Практические рекомендации:**
- Для независимых табличных данных: **10‑fold stratified CV**; для отчётной оценки — **repeated stratified k‑fold** (10 повторов 5‑fold).
- Для временных рядов: **TimeSeriesSplit** с расширяющимся или фиксированным окном в зависимости от задачи.
- При подборе гиперпараметров: начинайте с **random search** (100–200 итераций); если обучение очень дорогое (часы) — используйте **байесовскую оптимизацию** (20–50 итераций).
- Для итоговой честной оценки качества модели с настроенными гиперпараметрами применяйте **nested CV**. Это особенно важно в научных публикациях и при сравнении алгоритмов.

---

#### 5. Заключение и резюме главы

В этой главе мы изучили, как фундаментальная теория bias‑variance tradeoff объясняет поведение моделей, и как кросс‑валидация служит практическим инструментом для измерения обобщающей способности и выбора гиперпараметров.

- **Bias‑variance разложение** показало, что ожидаемая ошибка складывается из неустранимого шума, квадрата смещения (недообучение) и дисперсии (переобучение). С ростом сложности модели смещение падает, дисперсия растёт — существует оптимальная сложность.
- **Кросс‑валидация** позволяет оценить ошибку, избегая зависимости от единичного разбиения. Выбор схемы (k‑fold, LOOCV, стратифицированная, временная, групповая) определяется структурой данных.
- **Выбор гиперпараметров** (grid, random, Bayesian optimization) автоматизирует поиск лучшей конфигурации, балансируя вычислительный бюджет и качество.
- **Nested CV** даёт несмещённую оценку качества всей процедуры обучения, разделяя подбор гиперпараметров и финальную оценку.
- **Ансамбли** (bagging, boosting) целенаправленно уменьшают либо дисперсию, либо смещение, что выводит качество на новый уровень.

Для наглядной систематизации компромисса «смещение–дисперсия» в зависимости от сложности модели и объёма данных приведём итоговую таблицу.

| Характеристика | Простая модель (высокое bias, низкая var) | Сложная модель (низкий bias, высокая var) |
|----------------|-------------------------------------------|------------------------------------------|
| Ошибка на обучении | высокая | низкая (почти 0) |
| Ошибка на тесте | сравнима с обучением | значительно выше, чем на обучении |
| Добавление данных | слабо улучшает | значительно снижает ошибку |
| Регуляризация | не нужна или вредна | необходима, снижает дисперсию |
| Ансамблирование | неэффективно | bagging резко снижает дисперсию |
| Типичные методы | линейная регрессия, мелкое дерево | глубокое дерево, полином высокой степени, нейросеть |

Понимание этих взаимосвязей превращает интуитивный подбор моделей в осмысленную инженерную практику, позволяя быстро диагностировать проблемы и выбирать адекватные стратегии улучшения.

---

### Вопросы для самопроверки

**Для начинающих**
1. Из каких трёх компонентов складывается ожидаемая ошибка предсказания в регрессии?
2. В чём преимущество k‑fold кросс‑валидации перед однократным разбиением на обучение и тест?
3. Почему для прогнозирования временных рядов необходима специальная схема кросс‑валидации, а не обычный перемешанный k‑fold?
4. Что даёт вложенная кросс‑валидация по сравнению с обычной? В какой ситуации её применение особенно важно?
5. Какой метод поиска гиперпараметров вы выберете, если у модели 5 гиперпараметров и каждое обучение занимает 10 минут?
6. Почему бэггинг (bagging) снижает дисперсию предсказаний? К чему он стремится при увеличении числа базовых моделей?

**Для профессионалов**
1. Выведите разложение дисперсии ансамбля для бэггинга и объясните роль корреляции \(\rho\) между базовыми моделями.
2. Докажите, что обычная кросс‑валидация с последующим выбором лучшей модели даёт оптимистично смещённую оценку ошибки. В чём корень смещения?
3. Сравните Expected Improvement и Probability of Improvement как функции приобретения в байесовской оптимизации. Какая из них более «исследовательская»?
4. Как LOOCV-оценка связана с информационным критерием Акаике (AIC) для линейных моделей? Приведите формулу связи.
5. Предложите способ модификации nested CV для получения не только точечной оценки метрики, но и её доверительного интервала. Какие дополнительные предположения потребуются?

---

### Задания повышенной сложности

1. **Реализация LOOCV для линейной регрессии.** Напишите с нуля функцию, вычисляющую LOOCV-ошибку линейной регрессии через предсказанные значения \(\hat{y}\) и диагональ шляпной матрицы \(H_{ii}\). Сравните результат с прямым перебором по объектам и подтвердите совпадение с аналитической формулой PRESS.

2. **Эксперимент bias‑variance для полиномиальной регрессии.** Сгенерируйте данные с истинной зависимостью \(y = \sin(2\pi x) + \varepsilon\). Для степеней полинома \(d = 0, 1, \dots, 15\) повторите многократное обучение на бутстреп-выборках фиксированного размера. Оцените \(\text{Bias}^2\), \(\text{Variance}\) и полную ожидаемую ошибку в точке \(x_0\). Постройте кривые и подтвердите U‑образную форму total error.

3. **Байесовская оптимизация с нуля.** Используя `GaussianProcessRegressor` из sklearn, реализуйте байесовский поиск гиперпараметров \(C\) и \(\gamma\) для SVM без привлечения готовых библиотек (Optuna и т.п.). Реализуйте Expected Improvement. Сравните с `RandomizedSearchCV` по достигнутому качеству за фиксированное число итераций (например, 30).

4. **Смещение nested CV.** Формально докажите, что вложенная кросс‑валидация с \(K_{\text{out}}\) внешними фолдами и внутренней k‑fold настройкой является почти несмещённой оценкой ожидаемой ошибки процедуры «обучение + выбор гиперпараметров». Рассмотрите случай \(K_{\text{out}} = n\) (leave‑one‑out) и покажите связь с обычной CV.

5. **Сравнение CV на временном ряду.** Для авторегрессионного процесса AR(2) постройте прогнозную модель (линейная регрессия с лагами). Сравните ошибку прогноза на отложенном будущем периоде для трёх стратегий: (а) обычный перемешанный 5‑fold CV, (б) TimeSeriesSplit, (в) настоящая последовательная проверка (walk‑forward validation). Проиллюстрируйте эффект утечки данных и покажите, какая схема наиболее честно оценивает качество прогноза.